# Moonshot AI Kimi on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Moonshot AI's Kimi models. The two K2 variants are served by `bedrock-mantle` and are
built for long-context and agentic work, one of them explicitly a thinking model. Kimi
K3 is served by `bedrock-runtime` and not by `bedrock-mantle`, so the cells up to the
Converse section are about the K2 models and the last section is about K3.

**Models covered in this notebook**

| Model ID | Where it is served | Notes |
|---|---|---|
| `moonshotai.kimi-k2.5` | both endpoints | General purpose |
| `moonshotai.kimi-k2-thinking` | both, and renamed to `moonshot.kimi-k2-thinking` on `bedrock-runtime` | Reasoning-specialised variant |
| `moonshotai.kimi-k3` | `bedrock-runtime` only, through an inference profile | Latest generation. The last section measures which APIs and parameters it takes; this notebook does not probe its vision support |

## Which API? Per model, not per family.
The K2 models are served by the **OpenAI-compatible Chat Completions API** on the
`bedrock-mantle` endpoint, at the bare `/v1` path, and the Responses API returns
**400 "does not support this API"** for them — we prove that in §2 rather than asking
you to take it on trust. K3 takes both Chat Completions and Responses on
`bedrock-runtime`. The last section sends the same request to all three models on the
same paths, so the answer comes from the service rather than from a family rule.

## Self-contained, but see also
Everything you need is here. For deeper background on shared mechanics:
- **Auth (SigV4 (AWS Signature Version 4) + short-term API keys), the three URL paths,
  model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention / ZDR (zero data retention), CloudWatch
  namespace** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retry/backoff, service tiers, TTFT (time-to-first-token) measurement** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).


### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `parse_json_lenient` | parses the first complete JSON object out of model output, repairing truncated braces |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |
| `ttft` | times a streaming call: time-to-first-token and output frames/sec |
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `converse_reasoning` | the reasoning trace from a Converse response, or `""` |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `converse_text` | concatenates the text blocks of a Converse response — safer than `content[0]` |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import err, list_models, parse_json_lenient, post, safe_print, ttft

REGION = "us-east-1"

K25 = "moonshotai.kimi-k2.5"
THINKING = "moonshotai.kimi-k2-thinking"


# Chat-Completions families live at the BARE /v1 path — not /openai/v1
# (that prefix is only for gemma-4, gpt-5.x and grok). See ../00-foundations/01.
PREFIX = "/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)
print("models  :", [K25, THINKING])

base URL: https://bedrock-mantle.us-east-1.api.aws/v1
models  : ['moonshotai.kimi-k2.5', 'moonshotai.kimi-k2-thinking']


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "moonshotai.kimi-k2-thinking",
    "moonshotai.kimi-k2.5",
    "moonshotai.kimi-k3",
]

print(f"{'model (as this notebook names it)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only, runtime_only = [], []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    if not where["mantle"]:
        runtime_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

# Two different reasons a runtime id differs from the mantle one, and they need
# different code from you. A geo prefix leaves the id itself intact and means the
# model is only callable through an inference profile; a vendor rename does not,
# and means you cannot construct one id from the other.
prefixed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m and r.endswith(m)
]
renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m and not r.endswith(m)
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different vendor prefix, "
      f"{len(prefixed)} only through an inference profile.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
if runtime_only:
    print(f"   bedrock-runtime only: {runtime_only}")
    print("   The bedrock-mantle cells below do not apply to those; the last")
    print("   section of this notebook calls them on bedrock-runtime instead.")
if not mantle_only and not runtime_only:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as this notebook names it)      on runtime as                            endpoints
------------------------------------------------------------------------------------------------


moonshotai.kimi-k2-thinking            moonshot.kimi-k2-thinking                mantle, runtime


moonshotai.kimi-k2.5                   moonshotai.kimi-k2.5                     mantle, runtime


moonshotai.kimi-k3                     us.moonshotai.kimi-k3                    runtime



=> 3/3 of these are on bedrock-runtime; 1 under a different vendor prefix, 1 only through an inference profile.
   bedrock-runtime only: ['moonshotai.kimi-k3']
   The bedrock-mantle cells below do not apply to those; the last
   section of this notebook calls them on bedrock-runtime instead.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. First call

Auth is a short-term Bedrock API key minted from your ambient IAM credentials.
It expires within 12 hours and **cannot be refreshed** — mint a new one instead.
(`../00-foundations/01` shows the self-refreshing provider and the SigV4
alternative that needs no key at all.)

In [3]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# Build the client from a FRESH token — don't construct one at import time and
# reuse it for hours, because the baked-in key expires.
client = OpenAI(api_key=provide_token(region=REGION), base_url=BASE_URL)

completion = client.chat.completions.create(
    model=K25,
    messages=[
        {
            "role": "user",
            "content": (
                "Explain why long context windows matter for agents, in two sentences."
            ),
        }
    ],
    max_tokens=250,
)
# `content` can be None when the model spends the whole budget reasoning: the call
# succeeds with finish_reason="length" and no text. Check before printing -- this is
# the single most common surprise on this endpoint.
choice = completion.choices[0]
answer = choice.message.content or ""
if answer:
    print(answer)
else:
    print(f"(no text: finish_reason={choice.finish_reason!r} — raise max_tokens)")
print("\nusage:", completion.usage.model_dump_json())

 Long context windows allow agents to maintain extensive conversation history and reference complex documents without losing critical information, enabling more coherent and contextually aware interactions. This capability is essential for agents to handle multi-step tasks, reason across lengthy workflows, and build persistent understanding rather than treating each interaction in isolation.

usage: {"completion_tokens":58,"prompt_tokens":40,"total_tokens":98,"completion_tokens_details":null,"prompt_tokens_details":null}


## 2. Why Chat Completions and not Responses

AWS recommends the Responses API for new applications in general — but
availability is per-model. Probe both surfaces so the 400 is visible:

In [4]:
for api_name, path, body in [
    (
        "Chat Completions",
        f"{PREFIX}/chat/completions",
        {
            "model": K25,
            "messages": [{"role": "user", "content": "Reply OK"}],
            "max_tokens": 16,
        },
    ),
    (
        "Responses (/v1)",
        f"{PREFIX}/responses",
        {"model": K25, "input": "Reply OK", "max_output_tokens": 16},
    ),
    (
        "Responses (/openai/v1)",
        "/openai/v1/responses",
        {"model": K25, "input": "Reply OK", "max_output_tokens": 16},
    ),
]:
    code, data = post(path, body, region=REGION)
    print(f"  {api_name:24} -> HTTP {code} {'' if code == 200 else err(data)[:64]}")

  Chat Completions         -> HTTP 200 


  Responses (/v1)          -> HTTP 400 The model 'moonshotai.kimi-k2.5' does not support the '/v1/respo


  Responses (/openai/v1)   -> HTTP 400 The model 'moonshotai.kimi-k2.5' does not support the '/openai/v


Concrete consequences of being Chat-Completions-only:

- **You own the conversation history.** There is no `previous_response_id`
  server-side state on this API — send the full `messages` array each turn.
- **A reasoning trace may or may not come back, and the standard schema has
  nowhere to put it.** `reasoning_effort` is accepted family-wide. Where a
  trace IS returned it arrives in `choices[0].message.reasoning`, a sibling of
  `content` that is **not** part of the OpenAI Chat Completions schema, so a
  client written against the published spec drops it silently and you pay for
  tokens you never see. Whether this model returns one, and at which effort
  levels, is measured in §6 — read that table rather than this line.
  This bullet used to assert flatly that the trace is never returned, directly
  contradicting the probe further down the same notebook.
- Structured output uses `response_format`, not `text.format`.

## 3. Sampling parameters

This family accepts both `temperature` and `top_p`. That is *not* universal on
mantle: on Responses the GPT-5.5 and GPT-5.6 families accept `temperature` only at its default
`1.0` and refuse `top_p` outright, and newer Claude models reject both as
deprecated. So never share one sampling config across families — probe each one,
as the cell below does.

Which models refuse what has already changed twice during this collection's life:
Gemma 4 and Grok both tightened in August 2026 and have since been relaxed again.
Read the probe output, not this paragraph.

In [5]:
for label, extra in [
    ("temperature=0.7", {"temperature": 0.7}),
    ("temperature=0.0", {"temperature": 0.0}),
    ("top_p=0.95", {"top_p": 0.95}),
    ("both", {"temperature": 0.7, "top_p": 0.95}),
    ("max_tokens=1", {"max_tokens": 1}),
]:
    body = {
        "model": K25,
        "messages": [{"role": "user", "content": "Reply OK"}],
        "max_tokens": 16,
    }
    body.update(extra)
    code, data = post(f"{PREFIX}/chat/completions", body, region=REGION)
    print(f"  {label:18} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  temperature=0.7    -> HTTP 200 


  temperature=0.0    -> HTTP 200 


  top_p=0.95         -> HTTP 200 


  both               -> HTTP 200 


  max_tokens=1       -> HTTP 200 


Note `max_tokens=1` is accepted here. The Responses API enforces a minimum of
16 — another reason the two surfaces are not interchangeable.

## 4. Streaming

Chat Completions streams `data: {...}` SSE (server-sent events) frames carrying
`choices[0].delta.content`, terminated by `data: [DONE]`.

In [6]:
stream = client.chat.completions.create(
    model=K25,
    messages=[
        {
            "role": "user",
            "content": ("List four failure modes of long-running AI agents."),
        }
    ],
    max_tokens=300,
    stream=True,
)
chunks = 0
try:
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            chunks += 1
            print(delta, end="", flush=True)
except Exception as exc:
    # A stream can fail AFTER delivering part of the answer: a mid-stream
    # 5xx is not rare, and it has happened while building these notebooks.
    # Report what arrived instead of losing it - production code has to
    # decide whether a partial answer is usable or the call must be retried.
    print(f"\n[stream interrupted after {chunks} deltas: {type(exc).__name__}]")
print(f"\n\n[{chunks} content deltas received]")

 Here are four failure modes of long-running AI agents:

1. **Context drift / token limit overflow** — Over extended sessions, the agent's context window fills up, causing

 it to lose track of earlier instructions, goals, or conversation history, leading to inconsistent or degraded behavior.

2. **Accumulating error cascades** — Small mistakes in

 reasoning, perception, or tool use compound over time, with each step building on previous errors until the agent's outputs become unreliable or task-irrelevant.

3.

 **Goal misalignment / specification gaming** — The agent gradually optimizes for measurable proxy objectives rather than the intended goal, especially as it encounters edge cases or reward hacking opportunities

 during prolonged operation.

4. **Resource exhaustion or state corruption** — Memory leaks, unbounded log growth, persistent state inconsistency, or external API rate limits degrade performance

 or halt the agent outright after extended runtime.



[6 content deltas received]


## 5. Multi-turn — you manage the history

No server-side state on this API. Append each turn yourself.

In [7]:
messages = [
    {"role": "system", "content": "You are concise. Two sentences maximum."},
    {"role": "user", "content": "What is a tool-use loop?"},
]
first = client.chat.completions.create(model=K25, messages=messages, max_tokens=200)
print("assistant:", first.choices[0].message.content)

messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "What usually causes one to run away?"})

second = client.chat.completions.create(model=K25, messages=messages, max_tokens=200)
print("\nassistant:", second.choices[0].message.content)
print(
    f"\ninput tokens grew: {first.usage.prompt_tokens} -> {second.usage.prompt_tokens}"
)

assistant:  A tool-use loop is when an AI system repeatedly calls external tools (like APIs, calculators, or code interpreters) and feeds their outputs back into its own reasoning to solve complex tasks. This iterative cycle enables the AI to extend its capabilities beyond its base knowledge.



assistant:  Fear—whether of physical danger, emotional pain, or social rejection—is the most common trigger. The specific threat varies by context: predators, conflict, shame, or overwhelming pressure can all activate the flight response.

input tokens grew: 28 -> 98


That growth is the cost of client-side history. Families on the Responses API can
avoid it with `previous_response_id` (see `../03-google-gemma/`), at the price of
30-day server-side retention.

## 6. Reasoning effort

`reasoning_effort` is accepted. Two things about what comes back are worth getting
right, because the obvious reading of both is wrong.

**Where the trace goes.** On `bedrock-mantle` several models in the `/v1` families
return the reasoning in **`choices[0].message.reasoning`** — a sibling of
`content`, not part of it, and **not in the OpenAI specification**. SDK type hints
and any code written against the published schema will not surface it, so you can
pay for reasoning and discard it without noticing. Other models return nothing
there at all. It is a per-model fact, so the cell below asks rather than asserts.
(`../14-openai-gpt-oss/01` covers the same field for gpt-oss.)

**Whether effort changes spend.** Not measurably from one sample per level. Queue
time and the model's own choices dominate, so the token counts below will often
*not* rank in effort order. Measure repeatedly before drawing a cost conclusion.

In [8]:
print(f"{'effort':10} {'HTTP':>5} {'completion tok':>15} {'trace chars':>12}")
print("-" * 46)
efforts = {}
for effort in ("none", "low", "medium", "high"):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": K25,
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "I have 12 socks: 6 black, 4 blue, 2 grey. Drawing blind, how"
                        " many "
                        "must I take to guarantee a matching pair? Explain."
                    ),
                }
            ],
            # Budget well above the likely answer length. At 400 the cap itself
            # became the measurement: every row read 400 for some models, which
            # tells you nothing about effort.
            "max_tokens": 2000,
            "reasoning_effort": effort,
        },
        region=REGION,
    )
    usage = data.get("usage") or {}
    message = (data.get("choices") or [{}])[0].get("message") or {}
    # The non-standard field. `.get()` rather than indexing: plenty of models in
    # this family do not return it at all.
    trace = message.get("reasoning") or ""
    tokens = usage.get("completion_tokens", "-")
    efforts[effort] = (tokens, len(trace))
    print(f"  {effort:8} {code:>5} {tokens!s:>15} {len(trace):>12}")

# Both conclusions are DERIVED. The previous version of this cell asserted that the
# trace is never returned and that the token count rises with effort; for several
# models in this family both statements were false.
print()
if any(chars for _, chars in efforts.values()):
    print("=> The trace IS returned, in choices[0].message.reasoning -- a sibling of")
    print("   `content`, absent from the OpenAI schema. Read it explicitly or you")
    print("   are paying for tokens you never see.")
else:
    print("=> No trace in message.reasoning at any effort for this model: the")
    print("   thinking is billed and unreadable on this API.")
counts = [c for c, _ in efforts.values() if isinstance(c, int)]
if counts and counts != sorted(counts):
    print("=> The completion-token column is NOT in effort order. One sample per")
    print("   level cannot rank them -- do not read a cost curve off this table.")

effort      HTTP  completion tok  trace chars
----------------------------------------------


  none       200             192            0


  low        200             153            0


  medium     200             262            0


  high       200            1424         4592

=> The trace IS returned, in choices[0].message.reasoning -- a sibling of
   `content`, absent from the OpenAI schema. Read it explicitly or you
   are paying for tokens you never see.
=> The completion-token column is NOT in effort order. One sample per
   level cannot rank them -- do not read a cost curve off this table.


In [9]:
# The two variants split the work differently: k2-thinking is tuned for
# reasoning, k2.5 for general use. Same prompt, both models, compare spend.
RIDDLE = (
    "A rope ladder hangs over the side of a ship with rungs 30cm apart. "
    "The tide rises 15cm per hour. After 3 hours, how many rungs are "
    "underwater if 2 were underwater at the start? Explain."
)

for model in (K25, THINKING):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": RIDDLE}],
            # 3000, not 600. k2-thinking spent the whole 600 on its trace and
            # this cell published an empty answer for it, which reads as the
            # model failing rather than as too small a budget for a thinking
            # variant. That is the comparison the cell exists to make.
            "max_tokens": 3000,
            "reasoning_effort": "high",
        },
        region=REGION,
    )
    if code != 200:
        print(f"{model}: HTTP {code} {err(data)[:60]}")
        continue
    choice = data["choices"][0]
    text = (choice["message"]["content"] or "").strip()
    finish = choice.get("finish_reason")
    print(
        f"--- {model} | completion_tokens="
        f"{data['usage']['completion_tokens']} | finish={finish} ---"
    )
    if finish != "stop":
        print("    !! truncated at the budget, so this row measures the cap "
              "rather than the model.")
    print(text[:300] or "(no text returned)", "\n")

--- moonshotai.kimi-k2.5 | completion_tokens=713 | finish=stop ---
**Answer: 2 rungs are still underwater.**

**Explanation:**

The key to this puzzle is realizing that the ladder is attached to a **ship**, not to the harbor floor or a fixed dock. Since ships float, they rise with the tide as the water level increases. 

Therefore:
* As the tide rises 15 cm per hou 



--- moonshotai.kimi-k2-thinking | completion_tokens=2841 | finish=stop ---
**Answer: 2 rungs are still underwater after 3 hours.**

---

### Why the number doesn’t change

1. **The ship floats.**  
   A ship is a buoyant vessel; as the tide rises, the water level goes up relative to the seabed, but the ship is lifted up by the same amount.

2. **The ladder is attached to t 



## 7. Tool use (function calling)

Chat Completions nests the schema under `"function"` — unlike the Responses API,
which puts `name`/`parameters` at the top level. Same concept, different shape.

In [10]:
def lookup_inventory(sku: str, warehouse: str = "main") -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0, "C-300": 7}
    return {
        "sku": sku,
        "warehouse": warehouse,
        "quantity": stock.get(sku.upper(), 0),
        "in_stock": stock.get(sku.upper(), 0) > 0,
    }


tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_inventory",
            "description": "Look up stock level for a SKU.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sku": {"type": "string", "description": "SKU code, e.g. A-100"},
                    "warehouse": {"type": "string", "enum": ["main", "overflow"]},
                },
                "required": ["sku"],
            },
        },
    }
]

convo = [{"role": "user", "content": "Do we have SKU A-100 in stock?"}]

# `tool_choice="auto"` means the model MAY call a tool, not that it will. A
# reasoning-capable model can spend the budget thinking and return
# finish_reason="length" with no tool_calls. Retry instead of assuming.
msg = None
for attempt in range(1, 4):
    first = client.chat.completions.create(
        model=K25, messages=convo, tools=tools, tool_choice="auto", max_tokens=800
    )
    choice = first.choices[0]
    print(
        f"attempt {attempt}: finish_reason={choice.finish_reason!r} "
        f"tool_calls={len(choice.message.tool_calls or [])}"
    )
    if choice.message.tool_calls:
        msg = choice.message
        break
if msg is None:
    raise RuntimeError("no tool call after 3 attempts - raise max_tokens")
print(
    "tool_calls:",
    [(c.function.name, c.function.arguments) for c in (msg.tool_calls or [])],
)

if msg.tool_calls:
    convo.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:
        args = parse_json_lenient(call.function.arguments)
        result = lookup_inventory(**args)
        convo.append(
            {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)}
        )
    final = client.chat.completions.create(
        model=K25, messages=convo, tools=tools, max_tokens=200
    )
    print("\nfinal answer:", final.choices[0].message.content)

attempt 1: finish_reason='tool_calls' tool_calls=1
tool_calls: [('lookup_inventory', '{"sku": "A-100"}')]



final answer:  Yes, SKU A-100 is in stock! We currently have **42 units** available in the main warehouse.


### Forcing a specific tool

`tool_choice` can compel a named function. This is the most portable route to
strict structured output: the arguments *are* your JSON.

**But treat it as best-effort, not a guarantee.** In repeated testing about 1
call in 10 ignored the forced choice and returned prose with
`finish_reason="stop"`. Always check for the tool call and retry.

In [11]:
emit = [
    {
        "type": "function",
        "function": {
            "name": "emit_review",
            "description": "Return the structured review analysis.",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "sentiment": {
                        "type": "string",
                        "enum": ["positive", "neutral", "negative"],
                    },
                    "would_recommend": {"type": "boolean"},
                },
                "required": ["summary", "sentiment", "would_recommend"],
            },
        },
    }
]


def emit_review(prompt, model=K25, attempts=3):
    """Forced tool call, with a retry.

    IMPORTANT: forcing `tool_choice` is honoured *almost* always, not always.
    In repeated testing roughly 1 call in 10 came back with finish_reason="stop"
    and prose instead of a tool call. Production code must handle that, so this
    helper retries rather than indexing [0] and hoping.
    """
    for attempt in range(attempts):
        completion = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            tools=emit,
            tool_choice={"type": "function", "function": {"name": "emit_review"}},
            max_tokens=300,
        )
        choice = completion.choices[0]
        calls = choice.message.tool_calls or []
        if calls:
            if attempt:
                print(f"(succeeded on attempt {attempt + 1})")
            # parse_json_lenient, not json.loads: some models append characters
            # after a well-formed object even in strict modes.
            return parse_json_lenient(calls[0].function.arguments)
        print(
            f"attempt {attempt + 1}: no tool call "
            f"(finish_reason={choice.finish_reason}) — retrying"
        )
    raise RuntimeError("model would not emit the forced tool call")


review = emit_review("Review: 'Fast delivery, but the packaging arrived crushed.'")
print(json.dumps(review, indent=2))

{
  "sentiment": "neutral",
  "summary": "Customer appreciated the fast delivery but was disappointed with the condition of the packaging, which arrived crushed.",
  "would_recommend": false
}


## 8. Structured output with `response_format`

Two variants: loose `json_object`, and schema-enforced `json_schema`.

### Budget enough tokens, or you get nothing

A reasoning-capable model may spend most of its budget thinking before it emits
the opening brace. If `max_tokens` runs out first you get **HTTP 200 with empty
content** and `finish_reason="length"` - not an error, just nothing usable.
Always check `finish_reason` before parsing.

In [12]:
def json_object_call(prompt, max_tokens, model=K25):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "response_format": {"type": "json_object"},
        },
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content") or ""
    return code, choice.get("finish_reason"), content


PROMPT = "Give the capital and population of France as JSON."
for budget in (64, 600):
    code, finish, content = json_object_call(PROMPT, budget)
    print(
        f"max_tokens={budget:4} HTTP {code} finish={finish!s:8} "
        f"content_len={len(content)}"
    )
    # Check finish_reason FIRST, on its own. `finish == "length" and not
    # content.strip()` only catches the EMPTY truncation; a PARTIAL one -- non-empty
    # but cut mid-object -- fell through to parse_json_lenient and raised ValueError.
    # 09-minimax hit exactly that on a live run:
    #   ValueError: no JSON object found in 46 chars: '"capital": "Paris", ...}'
    # Two shapes of one failure, and the guard knew about one, in a cell whose whole
    # subject is checking finish_reason before parsing.
    if finish == "length":
        if not content.strip():
            print("    -> truncated before any JSON was emitted; raise max_tokens")
        else:
            print(f"    -> truncated MID-OBJECT after {len(content)} chars; the "
                  f"fragment is not parseable JSON and must not be parsed:")
            print(f"       {content.strip()[:120]!r}")
        continue
    if not content.strip():
        print("    -> no content and finish is not 'length'; inspect the response")
        continue
    try:
        print("    ->", parse_json_lenient(content))
    except ValueError as exc:
        print(f"    -> finish={finish} but the body is not JSON: {exc}")

max_tokens=  64 HTTP 200 finish=stop     content_len=42
    -> {'capital': 'Paris', 'population': 68042591}


max_tokens= 600 HTTP 200 finish=stop     content_len=74
    -> {'country': 'France', 'capital': 'Paris', 'population': 68042591}


In [13]:
schema = {
    "type": "object",
    "properties": {
        "country": {"type": "string"},
        "capital": {"type": "string"},
        "population_millions": {"type": "number"},
    },
    "required": ["country", "capital", "population_millions"],
    "additionalProperties": False,
}


def schema_call(budget: int):
    """One json_schema request at a given budget."""
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": K25,
            "messages": [{"role": "user", "content": "Describe France."}],
            "max_tokens": budget,
            "response_format": {
                "type": "json_schema",
                "json_schema": {"name": "country", "strict": True, "schema": schema},
            },
        },
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    return code, choice, (choice.get("message", {}) or {}).get("content")  # may be None


# Escalate rather than retrying once. On a reasoning-heavy model the trace can eat
# a raised budget too, and one retry is not enough to tell "needs more room" from
# "will never comply".
content, choice = None, {}
for budget in (250, 1000, 4000):
    code, choice, content = schema_call(budget)
    print(f"max_tokens={budget:5} HTTP {code} finish={choice.get('finish_reason')!s:8} "
          f"chars={len(content or '')}")
    if (content or "").strip():
        break

parsed = parse_json_lenient(content or "") if (content or "").strip() else {}
print("\nparsed:", json.dumps(parsed, indent=2))

if not parsed:
    # Empty is a BUDGET problem, not a schema violation. A sample that raises here
    # teaches nothing; the lesson is to check finish_reason and escalate, or to turn
    # reasoning off for extraction work where you do not need it.
    print(f"\nNo JSON at any budget (finish_reason={choice.get('finish_reason')!r}).")
    print("The reasoning trace consumed the whole budget before the opening brace.")
    print("Options: raise max_tokens further, or send reasoning_effort='none' --")
    print("extraction rarely needs the thinking, and it frees the budget for output.")
else:
    missing = {"country", "capital"} - set(parsed)
    if missing:
        # A NON-empty object missing required keys IS a strict-mode violation and
        # worth failing on: the schema promised those keys.
        raise ValueError(f"model omitted required keys: {sorted(missing)} in {parsed}")
    print("required keys present: country, capital")

max_tokens=  250 HTTP 200 finish=stop     chars=73

parsed: {
  "country": "France",
  "capital": "Paris",
  "population_millions": 67.5
}
required keys present: country, capital


**Always parse leniently.** Even in strict mode, some mantle models append
characters after a valid object (Gemma 4 does this in ~half of runs), which makes
a bare `json.loads()` raise on output that is otherwise fine.

## 9. Compare the models in this family

General-purpose versus reasoning-specialised, on one prompt.

In [14]:
task = "In one sentence, why do agents need a step budget?"

print(f"{'model':38} {'latency':>9} {'out tok':>8} {'trace':>7}  answer")
print("-" * 112)
for model in [K25, THINKING]:
    started = time.perf_counter()
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": task}],
            # The -thinking variant spends its budget on the trace first: at 160 this
            # column printed '' for it, which read as "the model said nothing".
            "max_tokens": 1500,
        },
        region=REGION,
    )
    elapsed = time.perf_counter() - started
    if code != 200:
        print(f"{model:38} {'-':>9} {'-':>8} {'-':>7}  HTTP {code}: {err(data)[:34]}")
        continue
    message = data["choices"][0]["message"]
    text = (message.get("content") or "").strip().replace("\n", " ")
    trace = message.get("reasoning") or ""
    print(
        f"{model:38} {elapsed:>8.2f}s "
        f"{data['usage']['completion_tokens']:>8} {len(trace):>7}  "
        f"{(text[:40] or '(empty - raise max_tokens)')!r}"
    )
print()
print("The trace column is why the budgets differ: -thinking spends tokens on")
print("reasoning before any answer text, and a cap that suits k2.5 starves it.")

model                                    latency  out tok   trace  answer
----------------------------------------------------------------------------------------------------------------


moonshotai.kimi-k2.5                       1.21s       26       0  'Agents need a step budget to prevent inf'


moonshotai.kimi-k2-thinking                6.83s      868    4573  'Agents need a step budget to prevent inf'

The trace column is why the budgets differ: -thinking spends tokens on
reasoning before any answer text, and a cap that suits k2.5 starves it.


## 10. Latency: TTFT and throughput

TTFT is dominated by *prefill* (the model reading your prompt) plus queue time.
Service tiers trade cost against queue priority.

Read the table below as **one sample, not a benchmark** — but do not expect the
three rows to look identical either. `flex` is deliberately deprioritised, so its
TTFT is usually the worst of the three by a clear margin; `default` and `priority`
sit close together on an idle account and separate under contention. A single
outlier in either direction is normal, and one `priority` call elsewhere in this
collection took 54 seconds. Take medians over many calls before quoting a number.
(`../00-foundations/03` has the full treatment.)

In [15]:
print(f"{'tier':10} {'TTFT (s)':>10} {'total (s)':>10} {'frames/s':>10}")
print("-" * 44)
for tier in ("default", "flex", "priority"):
    m = ttft(
        f"{PREFIX}/chat/completions",
        {
            "model": K25,
            "messages": [
                {
                    "role": "user",
                    "content": ("List four failure modes of long-running AI agents."),
                }
            ],
            "max_tokens": 200,
            "service_tier": tier,
        },
        region=REGION,
    )
    if m.get("error"):
        # Do NOT label every failure "tier not supported". A URLError or a timeout
        # is a transport problem and says nothing about whether the parameter is
        # accepted; reporting it as an unsupported feature invents a limitation
        # the service never claimed. Read the error before attributing a cause.
        detail = str(m["error"])
        if "service_tier" in detail or "unsupported" in detail.lower():
            cause = "tier refused by this model"
        else:
            cause = "transport error - retry; tells you nothing about tier support"
        print(f"{tier:10} {detail[:30]:>32}  ({cause})")
    else:
        print(
            f"{tier:10} {m['ttft_s']:>10.3f} {m['total_s']:>10.3f} "
            f"{m['frames_per_s']:>10.1f}"
        )

tier         TTFT (s)  total (s)   frames/s
--------------------------------------------


default         1.432      2.994        6.4


flex            2.139      2.139     6960.6


priority        1.264      2.233        7.2


## 11. Production hardening

Retries, cost attribution, and privacy. Mantle has **no RPM quota** — throttling
is token-based, and most models here have no published TPM (tokens per minute) quota
at all, so
capacity is fair-share. That makes retry-with-backoff mandatory, not optional.

In [16]:
code, project = post(
    "/v1/organization/projects",
    {
        "name": "kimi-samples",
        "tags": {"Application": "KimiDemo", "Environment": "Demo"},
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)

code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": K25,
        "messages": [{"role": "user", "content": "Reply OK"}],
        "max_tokens": 16,
    },
    region=REGION,
    headers={"OpenAI-Project": project_id},  # cost attribution
)
print("attributed call ->", code)

project: 200 proj_j2omhxrt...


attributed call -> 200


In [17]:
class KimiClient:
    """Demonstrates retry, attribution and structured-output patterns for this
    family on bedrock-mantle. A teaching pattern, not a production component:
    review and adapt it, and have it security-reviewed, before deployment."""

    def __init__(self, model=K25, region=REGION, tier="default", project=None):
        self.model, self.region, self.tier, self.project = model, region, tier, project

    def chat(self, messages, *, max_tokens=512, tools=None, schema=None, effort=None):
        body = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0.7,
            "service_tier": self.tier,
        }
        if tools:
            body["tools"] = tools
        if effort:
            body["reasoning_effort"] = effort
        if schema:
            body["response_format"] = {
                "type": "json_schema",
                "json_schema": {"name": "out", "strict": True, "schema": schema},
            }
        headers = {"OpenAI-Project": self.project} if self.project else None
        # post() retries 429 + 5xx with exponential backoff and jitter.
        code, data = post(
            f"{PREFIX}/chat/completions", body, region=self.region, headers=headers
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        return data

    @staticmethod
    def _choice(data: dict) -> dict:
        """First choice, without assuming the list is non-empty."""
        return (data.get("choices") or [{}])[0]

    def json(self, prompt, schema, *, max_tokens=512, **kw):
        """Structured call that survives a truncated first attempt.

        A reasoning-capable model can spend its whole budget thinking and return
        HTTP 200 with EMPTY content and finish_reason="length". Parsing that raises.
        So check finish_reason and escalate the budget once before giving up.
        """
        messages = [{"role": "user", "content": prompt}]
        for budget in (max_tokens, max_tokens * 4):
            data = self.chat(messages, schema=schema, max_tokens=budget, **kw)
            choice = self._choice(data)
            content = choice.get("message", {}).get("content") or ""
            if content.strip():
                return parse_json_lenient(content)
            if choice.get("finish_reason") != "length":
                break  # empty for some other reason - escalating will not help
        raise RuntimeError(
            f"no content after budget escalation to {max_tokens * 4} tokens "
            f"(finish_reason={self._choice(data).get('finish_reason')!r})"
        )


bot = KimiClient(tier="flex", project=project_id)
out = bot.json(
    "Name the largest ocean and its average depth in metres.",
    {
        "type": "object",
        "properties": {"ocean": {"type": "string"}, "avg_depth_m": {"type": "number"}},
        "required": ["ocean", "avg_depth_m"],
        "additionalProperties": False,
    },
)
print("structured result:", out)

structured result: {'ocean': 'Pacific Ocean', 'avg_depth_m': 4280}


In [18]:
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived demo project:", code, archived.get("status"))

archived demo project: 200 archived


## Regional footprint — check before you deploy

Which Regions carry a model is a per-model fact that changes as launches land, so this
asks the live catalogue rather than stating an answer that will quietly go stale.


In [19]:
# Ask both catalogues, not just this notebook's endpoint. An earlier version of
# this sweep asked bedrock-mantle alone and so printed "-" for kimi-k3, which
# bedrock-runtime serves in every Region below. Absent from one endpoint is not
# absent from Bedrock.
from bedrock import endpoints_for

regions = ("us-east-1", "us-east-2", "us-west-2", "eu-central-1")
WATCH = (
    ("kimi-k2.5", K25),
    ("kimi-k2-thinking", THINKING),
    ("kimi-k3", "moonshotai.kimi-k3"),
)

header = f"{'region':14}" + "".join(f"{label:>26}" for label, _ in WATCH)
print(header)
print("-" * len(header))
for region in regions:
    cells = []
    for _, model_id in WATCH:
        try:
            where = endpoints_for(model_id, region)
        except (RuntimeError, OSError) as exc:
            cells.append(type(exc).__name__)
            continue
        cells.append("+".join(n for n, there in where.items() if there) or "-")
    print(f"{region:14}" + "".join(f"{cell:>26}" for cell in cells))
print()
print("mantle = the endpoint the cells above use. runtime = Converse and the")
print("OpenAI-compatible paths on bedrock-runtime, where the id can differ; the")
print("endpoint cell near the top of this notebook prints the id to send.")

region                         kimi-k2.5          kimi-k2-thinking                   kimi-k3
--------------------------------------------------------------------------------------------


us-east-1                 mantle+runtime            mantle+runtime                   runtime


us-east-2                 mantle+runtime            mantle+runtime                   runtime


us-west-2                 mantle+runtime            mantle+runtime                   runtime


eu-central-1                           -                         -                   runtime

mantle = the endpoint the cells above use. runtime = Converse and the
OpenAI-compatible paths on bedrock-runtime, where the id can differ; the
endpoint cell near the top of this notebook prints the id to send.


## Gotchas — Moonshot AI Kimi on Bedrock

| Gotcha | Detail |
|---|---|
| Path prefix | Bare `/v1`, **not** `/openai/v1` (that's gemma-4 / gpt-5.x / grok) |
| Responses API | Per model, not per family: **400** for the K2 models, accepted for `kimi-k3` on `bedrock-runtime`. The last section probes all three |
| History | No `previous_response_id`; you send `messages` every turn |
| Reasoning trace | Per model: several models here return it in `choices[0].message.reasoning`, which is **not** in the OpenAI schema; others return nothing. §6 probes it |
| Strict JSON | Parse leniently — models can append text after a valid object |
| Sampling | Per model **and** per API: `temperature` and `top_p` are both fine for the K2 models on mantle; for `kimi-k3` both OpenAI-compatible paths take any `temperature` and accept `top_p` at one value only, and Converse refuses both fields outright |
| `max_tokens` | 1 is valid here; Responses API demands ≥16 |
| `content` can be `None` | Check `finish_reason` before slicing/parsing content |
| Inference profile | `kimi-k3` is `INFERENCE_PROFILE`-only: the bare id is refused, and which prefix works is Region-specific — `us.` in us-east-1, `global.` in eu-central-1. Let `resolve_runtime_id()` pick |
| Prompt caching | `cachePoint` on Converse is refused for all three, and with two different error types — one of them `AccessDeniedException`, which is not a permissions problem. Explicit caching on the OpenAI paths works for `kimi-k3` and is **silently ignored** for `kimi-k2.5`: HTTP 200 either way, so read `usage`, never the status |
| Quotas | No RPM quota; most models have no published TPM — retry with backoff |
| `reserved` tier | Rejected as a parameter; arranged via your account team |
| CloudWatch | Metrics land in `AWS/BedrockMantle`, not `AWS/Bedrock`, for the mantle calls |
| Thinking variant | `kimi-k2-thinking` spends its budget on the trace first — a cap that suits `kimi-k2.5` returns HTTP 200 and empty `content` |
| Region | Per model and per endpoint, and it moves — the probe above asks both catalogues. The model card's regional table is authoritative |

## Where next
- Same API shape: `../05-deepseek/`, `../09-minimax/`, `../04-qwen/`
- Different API shape: `../03-google-gemma/` (Responses),
  `../02-anthropic-claude/` (Messages), `../01-openai-gpt/` (web search, caching)
- Shared mechanics: `../00-foundations/`


## Also on `bedrock-runtime`? Kimi

Watch the **provider prefix**: this model is `moonshotai.kimi-k2-thinking` on `bedrock-mantle` and `moonshot.kimi-k2-thinking` on `bedrock-runtime` - the vendor namespace itself differs. `kimi-k2.5` keeps `moonshotai.` on both.

`endpoints_for()` asks both catalogues rather than trusting a table, so the cell
below tells you today's answer. Converse is worth reaching for when you want one
request shape across providers, or a feature that only `bedrock-runtime` carries.


In [20]:
from bedrock import (
    converse,
    converse_reasoning,
    endpoints_for,
    resolve_runtime_id,
)

MANTLE_ID = "moonshotai.kimi-k2-thinking"
RUNTIME_ID = "moonshot.kimi-k2-thinking"

print("endpoint availability:", endpoints_for(MANTLE_ID))
print("mantle model id :", MANTLE_ID)
print("runtime model id:", RUNTIME_ID)
resolved = resolve_runtime_id(RUNTIME_ID)
print("converse sends  :", resolved)
if resolved != RUNTIME_ID:
    print("                  ^ resolved for you; the form above would be rejected")

# The same question, through Converse. Note the shape: content is a LIST of
# blocks rather than a string, and the token budget lives in inferenceConfig.
# The budget is generous on purpose - a reasoning model spends it on the trace
# first and returns no text block at all if it runs out.
text, response = converse(
    RUNTIME_ID,
    [
        {
            "role": "user",
            "content": [
                {"text": "Name one benefit of idempotency. One sentence."}
            ],
        }
    ],
    max_tokens=400,
    system="You are terse.",
)

error = (response.get("error") or {}).get("message")
if error:
    print("\ncall failed:", error[:200])
else:
    reasoning = converse_reasoning(response)
    print("\nstop reason:", response.get("stopReason"))
    print("tokens     :", response.get("usage", {}).get("totalTokens"))
    if reasoning:
        print(f"reasoning  : {len(reasoning)} chars (returned in a "
              "reasoningContent block, before the text)")
    if text.strip():
        print("answer     :", text.strip()[:200])
    else:
        # Empty text is NOT the same as a failed call. Say which it is.
        print("answer     : (none - the budget went to reasoning; raise max_tokens)")


endpoint availability: {'mantle': True, 'runtime': True}
mantle model id : moonshotai.kimi-k2-thinking
runtime model id: moonshot.kimi-k2-thinking


converse sends  : moonshot.kimi-k2-thinking



stop reason: end_turn
tokens     : 159
reasoning  : 612 chars (returned in a reasoningContent block, before the text)
answer     : Idempotency allows safe retry of failed operations without causing duplicate or unintended side effects.


## Converse in earnest — the tool loop, provider parameters, and caching

The earlier endpoint section proved this model answers through Converse. That is
the easy part. This section does the three things you actually need on
`bedrock-runtime`, because each differs from the `bedrock-mantle` equivalent:

1. **A complete tool round trip** — `toolUse` out, `toolResult` back in. Getting a
   tool *call* is half the job; feeding the result back is where the shapes bite.
2. **`additionalModelRequestFields`** — Converse normalises the common fields, so
   anything provider-specific goes through this escape hatch.
3. **`cachePoint`** — prompt caching is a first-class Converse block, and support
   for it is per model rather than universal.


In [21]:
from bedrock import converse_text, converse_tool_uses, resolve_runtime_id, runtime_client

RUNTIME_ID = "moonshot.kimi-k2-thinking"
runtime = runtime_client(REGION)
resolved = resolve_runtime_id(RUNTIME_ID, REGION)

# Converse tool shape: toolSpec, and the JSON Schema nests under inputSchema.json.
# This is NOT the OpenAI shape - there is no {"type": "function"} wrapper.
WEATHER_TOOL = {
    "toolSpec": {
        "name": "get_weather",
        "description": "Current weather for a city",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            }
        },
    }
}

history = [
    {"role": "user", "content": [{"text": "What is the weather in Singapore? Use the tool."}]}
]
first = runtime.converse(
    modelId=resolved,
    messages=history,
    toolConfig={"tools": [WEATHER_TOOL]},
    inferenceConfig={"maxTokens": 500},
)
print("turn 1 stop reason:", first.get("stopReason"))
print("turn 1 blocks     :", [next(iter(b)) for b in first["output"]["message"]["content"]])

uses = converse_tool_uses(first)
if not uses:
    print("no tool call this run - tool_choice defaults to the model's discretion;")
    print("retry, or set toolConfig['toolChoice'] to compel one.")
else:
    use = uses[0]
    print(f"tool call         : {use['name']}({use['input']})")
    # Validate before acting on it. A malformed call still reports tool_use.
    city = str(use["input"].get("city", "")).lower()
    print("arguments valid   :", "yes" if "singapore" in city else f"NO ({use['input']})")

    # Echo the assistant turn back VERBATIM, then answer with a toolResult whose
    # toolUseId matches. Dropping either breaks the loop with a 400.
    history.append(first["output"]["message"])
    history.append(
        {
            "role": "user",
            "content": [
                {
                    "toolResult": {
                        "toolUseId": use["toolUseId"],
                        "content": [{"json": {"tempC": 31, "conditions": "humid"}}],
                    }
                }
            ],
        }
    )
    second = runtime.converse(
        modelId=resolved,
        messages=history,
        toolConfig={"tools": [WEATHER_TOOL]},
        inferenceConfig={"maxTokens": 300},
    )
    print("turn 2 stop reason:", second.get("stopReason"))
    print("final answer      :", converse_text(second).strip()[:160])


turn 1 stop reason: tool_use
turn 1 blocks     : ['reasoningContent', 'toolUse']
tool call         : get_weather({'city': 'Singapore'})
arguments valid   : yes


turn 2 stop reason: end_turn
final answer      : <think> The weather in Singapore is currently **31°C** with **humid conditions**.


In [22]:
# Converse normalises maxTokens, temperature, topP and stopSequences. Anything
# provider-specific goes through additionalModelRequestFields, unvalidated by
# Converse and passed to the provider as-is. That makes it powerful and sharp:
# a key this model does not recognise is a 400, not a silent no-op.
from bedrock import converse_reasoning

PUZZLE = (
    "A bat and ball cost $1.10 together. The bat costs $1.00 more than the ball. "
    "How much is the ball?"
)

for label, extra in [
    ("no extra fields", None),
    ("provider fields", {"reasoning_effort": "low"}),
]:
    kwargs = {"additionalModelRequestFields": extra} if extra else {}
    try:
        response = runtime.converse(
            modelId=resolved,
            messages=[{"role": "user", "content": [{"text": PUZZLE}]}],
            inferenceConfig={"maxTokens": 900},
            **kwargs,
        )
    except Exception as exc:
        print(f"{label:<16} {type(exc).__name__}: {str(exc)[-90:]}")
        continue
    blocks = [next(iter(b)) for b in response["output"]["message"]["content"]]
    trace = converse_reasoning(response)
    answer = converse_text(response).strip().replace("\n", " ")
    print(f"{label:<16} out={response['usage']['outputTokens']:>4} blocks={blocks}")
    print(f"{'':<16} reasoning={len(trace)} chars | {answer[:70]}")

print()
print("Note whether a reasoningContent block appears above. Some models return the")
print("trace as a typed block on Converse and some do not, so read the blocks")
print("rather than assuming - and never index content[0].")


no extra fields  out= 327 blocks=['reasoningContent', 'text']
                 reasoning=664 chars | The ball costs **$0.05** (5 cents).  Here's the breakdown: - Ball = $0


provider fields  out= 436 blocks=['reasoningContent', 'text']
                 reasoning=826 chars | The ball costs **$0.05** (5 cents).  Here's the math: - Let ball = x -

Note whether a reasoningContent block appears above. Some models return the
trace as a typed block on Converse and some do not, so read the blocks
rather than assuming - and never index content[0].


In [23]:
# cachePoint is a Converse block, but support for it is per model rather than
# universal. Ask before designing around it.
HANDBOOK = "You are a support handbook. " + (
    "Retries: use exponential backoff with full jitter, cap at 16 seconds. " * 160
)

try:
    usage = runtime.converse(
        modelId=resolved,
        system=[{"text": HANDBOOK}, {"cachePoint": {"type": "default"}}],
        messages=[{"role": "user", "content": [{"text": "One line: the retry policy?"}]}],
        inferenceConfig={"maxTokens": 60},
    )["usage"]
    print("cachePoint accepted:", {k: v for k, v in usage.items() if "cache" in k.lower()})
except Exception as exc:
    print(f"cachePoint -> {type(exc).__name__}")
    print(f"    {str(exc)[-140:]}")
    print()
    print("Not every model supports prompt caching on Converse. Note the exception")
    print("type: this surfaces as an access or validation error rather than a clear")
    print("'unsupported feature' message, which is easy to misread as a permissions")
    print("problem. Probe it once per model instead of assuming it is available.")

# The same call without the cachePoint block works, so caching is the only part
# that is unavailable.
usage = runtime.converse(
    modelId=resolved,
    system=[{"text": "You are terse."}],
    messages=[{"role": "user", "content": [{"text": "One line: why use backoff?"}]}],
    inferenceConfig={"maxTokens": 60},
)["usage"]
print()
print("same call without cachePoint -> OK, tokens:", usage["totalTokens"])


cachePoint -> AccessDeniedException
    nverse operation: You invoked an unsupported model or your request did not allow prompt caching. See the documentation for more information.

Not every model supports prompt caching on Converse. Note the exception
type: this surfaces as an access or validation error rather than a clear
'unsupported feature' message, which is easy to misread as a permissions
problem. Probe it once per model instead of assuming it is available.



same call without cachePoint -> OK, tokens: 82


### What this section adds over the endpoint check above

- **The tool loop is the part that bites.** `toolSpec` is not the OpenAI shape,
  the JSON Schema nests under `inputSchema.json`, and the second turn must echo the
  assistant message back verbatim alongside a `toolResult` whose `toolUseId`
  matches. Miss any of that and you get a 400.
- **`additionalModelRequestFields` is unvalidated by Converse.** It is the only way
  to reach provider-specific behaviour, and a key the model does not recognise
  fails the call rather than being ignored.
- **Feature support is per model, not per endpoint.** Read the output above rather
  than carrying an assumption over from another family.


## Kimi K3 — a third model, on the other endpoint

`moonshotai.kimi-k3` is served by `bedrock-runtime` and does not appear in the
`bedrock-mantle` catalogue in any Region the sweep above checked, so nothing earlier in
this notebook reaches it.

The gotchas table above used to say the Responses API returns **400 for this family**.
That was measured on the K2 models, and it is false for K3 — the defect shape this
collection keeps finding, a fact keyed on the family rather than on the model and the
API together. Two more behaviours have no family-level answer at all. So the cells below
send the same request to each model and print what came back:

1. **Which API** is per model: K3 answers on the Responses API, where both K2 models
   return 400 on that same path.
2. **Sampling** is per model and per surface: the OpenAI-compatible paths take any
   `temperature` and accept `top_p` at one value only, and Converse refuses both
   fields.
3. **Prompt caching** is per model, and a model that does not support it returns
   HTTP 200 while caching nothing, so the only honest test reads `usage`.

Keep the ordering in mind: the id you send comes first, because every probe below fails
in a confusing way if you send the bare model id.

In [24]:
# K3 is INFERENCE_PROFILE-only, which the catalogue says outright. That single
# field is why the bare id is refused, and it is worth reading before you debug a
# ValidationException that mentions on-demand throughput.
from bedrock import inference_profiles, list_models, resolve_runtime_id, runtime_models

K3 = "moonshotai.kimi-k3"
REGIONS = ("us-east-1", "us-west-2", "eu-central-1")

header = (f"{'region':14}{'mantle':>8}{'runtime':>9}  "
          f"{'k3 inference profiles in Region':52}resolve_runtime_id()")
print(header)
print("-" * len(header))
for region in REGIONS:
    on_mantle = K3 in list_models(region)
    entry = runtime_models(region).get(K3)
    profiles = sorted(p for p in inference_profiles(region) if K3 in p)
    print(f"{region:14}{('yes' if on_mantle else '-'):>8}"
          f"{('yes' if entry else '-'):>9}  {(', '.join(profiles) or 'none'):52}"
          f"{resolve_runtime_id(K3, region)}")

entry = runtime_models(REGION).get(K3) or {}
# Some catalogue entries carry the inference types under the versioned id.
types = sorted(entry.get("id_infer") or entry.get("infer") or [])
print()
print(f"inference types on bedrock-runtime in {REGION}: {types}")
print(f"=> ON_DEMAND offered: {'ON_DEMAND' in types}. Without ON_DEMAND the bare")
print("   id is not callable at all, and which profile prefix exists depends on")
print("   the Region, as the table shows. The next cell sends each form.")

region          mantle  runtime  k3 inference profiles in Region                     resolve_runtime_id()
---------------------------------------------------------------------------------------------------------
us-east-1            -      yes  global.moonshotai.kimi-k3, us.moonshotai.kimi-k3    us.moonshotai.kimi-k3


us-west-2            -      yes  global.moonshotai.kimi-k3, us.moonshotai.kimi-k3    us.moonshotai.kimi-k3


eu-central-1         -      yes  global.moonshotai.kimi-k3                           global.moonshotai.kimi-k3

inference types on bedrock-runtime in us-east-1: ['INFERENCE_PROFILE']
=> ON_DEMAND offered: False. Without ON_DEMAND the bare
   id is not callable at all, and which profile prefix exists depends on
   the Region, as the table shows. The next cell sends each form.


In [25]:
# What each id form actually does, in a Region that has both profiles and one
# that has only the global profile. resolve=False sends the id verbatim; the
# helper's job is exactly to save you from choosing here.
from bedrock import converse, err

ASK = [{"role": "user", "content": [{"text": "Reply with the single word OK."}]}]
FORMS = (K3, f"us.{K3}", f"global.{K3}")

print(f"{'region':14}{'id sent to Converse':30}result")
print("-" * 70)
accepted, messages = {}, {}
for region in ("us-east-1", "eu-central-1"):
    for form in FORMS:
        text, response = converse(form, ASK, region=region, max_tokens=64,
                                  resolve=False)
        error = response.get("error") or {}
        if error:
            # converse() returns the botocore error rather than an HTTP status,
            # so print the code the caller would catch, not an invented status.
            result = error.get("code", "error")
            messages[err(error, 200)] = messages.get(err(error, 200), 0) + 1
        else:
            result = f"200, stopReason={response['stopReason']}"
            accepted.setdefault(region, []).append(form)
        print(f"{region:14}{form:30}{result}")

print()
print("the refusals in full, which is where the instruction is:")
for message, count in messages.items():
    print(f"  ({count}x) {message}")
print()
for region in ("us-east-1", "eu-central-1"):
    print(f"=> {region}: accepted {accepted.get(region, [])}")
print("   The bare id is refused in both, and the us. profile does not exist in")
print("   eu-central-1. resolve_runtime_id() returns a form from this accepted set")
print("   for each Region, which is why the cells above pass it the bare id.")

region        id sent to Converse           result
----------------------------------------------------------------------
us-east-1     moonshotai.kimi-k3            ValidationException


us-east-1     us.moonshotai.kimi-k3         200, stopReason=max_tokens


us-east-1     global.moonshotai.kimi-k3     200, stopReason=end_turn


eu-central-1  moonshotai.kimi-k3            ValidationException


eu-central-1  us.moonshotai.kimi-k3         ValidationException


eu-central-1  global.moonshotai.kimi-k3     200, stopReason=end_turn

the refusals in full, which is where the instruction is:
  (2x) Invocation of model ID moonshotai.kimi-k3 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.
  (1x) The provided model identifier is invalid.

=> us-east-1: accepted ['us.moonshotai.kimi-k3', 'global.moonshotai.kimi-k3']
=> eu-central-1: accepted ['global.moonshotai.kimi-k3']
   The bare id is refused in both, and the us. profile does not exist in
   eu-central-1. resolve_runtime_id() returns a form from this accepted set
   for each Region, which is why the cells above pass it the bare id.


In [26]:
# Which API reaches which model, on one endpoint, in one Region. The comparison
# is the point: same path, same body shape, different answers per model.
from bedrock import ok, runtime_post

K3_RUNTIME = resolve_runtime_id(K3, REGION)
MODELS = [K3_RUNTIME, "moonshotai.kimi-k2.5", "moonshot.kimi-k2-thinking"]
QUESTION = "Reply with the single word OK."

print(f"{'model on bedrock-runtime':30}{'/openai/v1/responses':24}"
      f"/openai/v1/chat/completions")
print("-" * 104)
refusals = {}
reached = {"/openai/v1/responses": [], "/openai/v1/chat/completions": []}
for model in MODELS:
    row = []
    for path, body in (
        ("/openai/v1/responses",
         {"model": model, "input": QUESTION, "max_output_tokens": 64}),
        ("/openai/v1/chat/completions",
         {"model": model, "messages": [{"role": "user", "content": QUESTION}],
          "max_tokens": 64}),
    ):
        code, payload = runtime_post(path, body, region=REGION)
        if ok(code, payload):
            reached[path].append(model)
            row.append("200")
        else:
            row.append(str(code))
            refusals[(model, path)] = err(payload, 120)
    print(f"{model:30}{row[0]:24}{row[1]}")

print()
for (model, path), message in refusals.items():
    print(f"{path} refused {model}:")
    print(f"    {message}")
print()
for path, models in reached.items():
    print(f"=> {path} reached {len(models)}/{len(MODELS)}: {models}")
print("   So 'this family is Chat Completions only' was a statement about two")
print("   models, not about the family. Ask per model and per path.")

model on bedrock-runtime      /openai/v1/responses    /openai/v1/chat/completions
--------------------------------------------------------------------------------------------------------


us.moonshotai.kimi-k3         200                     200


moonshotai.kimi-k2.5          400                     200


moonshot.kimi-k2-thinking     400                     200

/openai/v1/responses refused moonshotai.kimi-k2.5:
    The model 'moonshotai.kimi-k2.5' does not support the '/openai/v1/responses' API
/openai/v1/responses refused moonshot.kimi-k2-thinking:
    The model 'moonshot.kimi-k2-thinking' does not support the '/openai/v1/responses' API

=> /openai/v1/responses reached 1/3: ['us.moonshotai.kimi-k3']
=> /openai/v1/chat/completions reached 3/3: ['us.moonshotai.kimi-k3', 'moonshotai.kimi-k2.5', 'moonshot.kimi-k2-thinking']
   So 'this family is Chat Completions only' was a statement about two
   models, not about the family. Ask per model and per path.


In [27]:
# Sampling limits, on every surface this model serves, for the same model. The
# service names the value it will accept, which is more than any table here could
# keep current.
from bedrock import runtime_client

runtime = runtime_client(REGION)
OPENAI_PATHS = {
    "/openai/v1/responses": lambda extra: {
        "model": K3_RUNTIME, "input": QUESTION, "max_output_tokens": 64, **extra},
    "/openai/v1/chat/completions": lambda extra: {
        "model": K3_RUNTIME, "messages": [{"role": "user", "content": QUESTION}],
        "max_tokens": 64, **extra},
}
SENT = (("(none)", {}), ("temperature=0.2", {"temperature": 0.2}),
        ("top_p=0.9", {"top_p": 0.9}), ("top_p=0.95", {"top_p": 0.95}))
accepts = {}

print(f"{'surface':30}{'parameter':20}result")
print("-" * 104)
for path, build in OPENAI_PATHS.items():
    for label, extra in SENT:
        code, payload = runtime_post(path, build(extra), region=REGION)
        accepts[(path, label)] = ok(code, payload)
        # Do not truncate this message: it names the one value the model accepts,
        # and a cut-off number is worse than no number.
        print(f"{path:30}{label:20}"
              f"{'200' if accepts[(path, label)] else str(code) + ' ' + err(payload, 120)}")

for label, config in (("(none)", {}), ("temperature=0.2", {"temperature": 0.2}),
                      ("topP=0.9", {"topP": 0.9}), ("topP=0.95", {"topP": 0.95})):
    try:
        response = runtime.converse(
            modelId=K3_RUNTIME,
            messages=[{"role": "user", "content": [{"text": QUESTION}]}],
            inferenceConfig={"maxTokens": 64, **config},
        )
        accepts[("Converse", label)] = True
        print(f"{'Converse':30}{label:20}200 stopReason={response['stopReason']}")
    except Exception as exc:
        accepts[("Converse", label)] = False
        detail = (getattr(exc, "response", {}) or {}).get("Error", {}).get("Message")
        print(f"{'Converse':30}{label:20}{type(exc).__name__}: "
              f"{detail or str(exc)[-90:]}")

print()
surfaces = list(OPENAI_PATHS) + ["Converse"]
for parameter in ("temperature=0.2", "top_p=0.9", "top_p=0.95"):
    keys = [(s, parameter if s != "Converse" else parameter.replace("top_p", "topP"))
            for s in surfaces]
    taken = [s for s, label in keys if accepts.get((s, label))]
    print(f"=> {parameter:16} accepted on: {taken or 'no surface here'}")
print("   A model that takes a parameter on one surface and refuses it on another")
print("   cannot be described by a per-model rule alone. Send neither field unless")
print("   you have probed the surface you are on.")

surface                       parameter           result
--------------------------------------------------------------------------------------------------------


/openai/v1/responses          (none)              200


/openai/v1/responses          temperature=0.2     200


/openai/v1/responses          top_p=0.9           400 Invalid value: 0.9. This model accepts 'top_p' only with the value 0.95.


/openai/v1/responses          top_p=0.95          200


...

/openai/v1/chat/completions   temperature=0.2     200


/openai/v1/chat/completions   top_p=0.9           400 Invalid value: 0.9. This model accepts 'top_p' only with the value 0.95.


/openai/v1/chat/completions   top_p=0.95          200


Converse                      (none)              200 stopReason=end_turn
Converse                      temperature=0.2     ValidationException: This model doesn't support the temperature field. Remove temperature and try again.
Converse                      topP=0.9            ValidationException: This model doesn't support the topP field. Remove topP and try again.


Converse                      topP=0.95           ValidationException: This model doesn't support the topP field. Remove topP and try again.

=> temperature=0.2  accepted on: ['/openai/v1/responses', '/openai/v1/chat/completions']
=> top_p=0.9        accepted on: no surface here
=> top_p=0.95       accepted on: ['/openai/v1/responses', '/openai/v1/chat/completions']
   A model that takes a parameter on one surface and refuses it on another
   cannot be described by a per-model rule alone. Send neither field unless
   you have probed the surface you are on.


In [28]:
# Explicit prompt caching on the OpenAI-compatible path: request-level
# prompt_cache_options plus a per-block prompt_cache_breakpoint. The dangerous
# part is the failure mode - a model that does not support it still answers 200.
import uuid


def cached_call(model, prefix, question):
    body = {
        "model": model,
        "max_tokens": 48,
        "prompt_cache_options": {"mode": "explicit", "ttl": "30m"},
        "messages": [
            {"role": "system", "content": [
                {"type": "text", "text": prefix,
                 "prompt_cache_breakpoint": {"mode": "explicit"}}]},
            {"role": "user", "content": question},
        ],
    }
    code, payload = runtime_post("/openai/v1/chat/completions", body, region=REGION)
    if not ok(code, payload):
        return None, f"{code} {err(payload, 90)}"
    usage = payload.get("usage") or {}
    return (usage.get("prompt_tokens_details") or {}), None


print(f"{'model':30}{'call':8}prompt_tokens_details")
print("-" * 104)
observed = {}
for model in (K3_RUNTIME, "moonshotai.kimi-k2.5"):
    # A cache entry outlives the run, so a fixed prefix would hit on the first
    # call and the write would never be visible. Make the prefix unique per run.
    # HANDBOOK is the ~2,600-token block defined in the cachePoint cell above;
    # explicit caching has a minimum prefix length and a short prompt silently
    # caches nothing.
    prefix = f"Handbook revision {uuid.uuid4()}. " + HANDBOOK
    for call in (1, 2):
        detail, failure = cached_call(model, prefix, "One line: the retry policy?")
        observed.setdefault(model, []).append(detail or {})
        print(f"{model:30}{call:<8}{failure or detail}")

print()
for model, (first, second) in observed.items():
    wrote = bool(first.get("cache_write_tokens"))
    read = bool(second.get("cached_tokens"))
    verdict = ("wrote then read the cache" if wrote and read
               else "no cache write or read reported")
    print(f"{model:30} wrote={wrote} read={read} -> {verdict}")
print()
print("the same feature through Converse, where it is a typed block:")
cache_point = {}
for model in (K3_RUNTIME, "moonshotai.kimi-k2.5", "moonshot.kimi-k2-thinking"):
    try:
        usage = runtime.converse(
            modelId=model,
            system=[{"text": HANDBOOK}, {"cachePoint": {"type": "default"}}],
            messages=[{"role": "user",
                       "content": [{"text": "One line: the retry policy?"}]}],
            inferenceConfig={"maxTokens": 64},
        )["usage"]
        cache_point[model] = "200"
        reported = {k: v for k, v in usage.items() if "ache" in k.lower()}
        print(f"  {model:30} 200 {reported}")
    except Exception as exc:
        cache_point[model] = type(exc).__name__
        detail = (getattr(exc, "response", {}) or {}).get("Error", {}).get("Message")
        print(f"  {model:30} {type(exc).__name__}: "
              f"{(detail or str(exc))[:64]}")

print()
accepted = [m for m, outcome in cache_point.items() if outcome == "200"]
kinds = sorted({outcome for outcome in cache_point.values() if outcome != "200"})
print(f"=> cachePoint accepted by: {accepted or 'none of these models'}")
print(f"=> refused with {len(kinds)} different error type(s): {kinds}")
print("   AccessDeniedException reads like a permissions problem and is not one")
print("   here, which is why this probe prints the type rather than the status.")
print("=> on the OpenAI-compatible path, both models returned HTTP 200 for both")
print("   calls and only usage tells them apart, so a caching integration that")
print("   checks status codes reports success while paying full price.")

model                         call    prompt_tokens_details
--------------------------------------------------------------------------------------------------------


us.moonshotai.kimi-k3         1       {'audio_tokens': 0, 'cache_write_tokens': 2672, 'cached_tokens': 0}


us.moonshotai.kimi-k3         2       {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 2672}


moonshotai.kimi-k2.5          1       {}


moonshotai.kimi-k2.5          2       {}

us.moonshotai.kimi-k3          wrote=True read=True -> wrote then read the cache
moonshotai.kimi-k2.5           wrote=False read=False -> no cache write or read reported

the same feature through Converse, where it is a typed block:
  us.moonshotai.kimi-k3          ValidationException: This model doesn't support the cachePoint field. Remove cachePoi
  moonshotai.kimi-k2.5           AccessDeniedException: You invoked an unsupported model or your request did not allow p
  moonshot.kimi-k2-thinking      AccessDeniedException: You invoked an unsupported model or your request did not allow p

=> cachePoint accepted by: none of these models
=> refused with 2 different error type(s): ['AccessDeniedException', 'ValidationException']
   AccessDeniedException reads like a permissions problem and is not one
   here, which is why this probe prints the type rather than the status.
=> on the OpenAI-compatible path, both models returned HTTP 200 for both
   c

### What this section adds over the K2 cells

- **A runtime-only model in a mantle notebook.** The endpoint cell near the top and
  the regional sweep both ask two catalogues now, because asking one printed `-` for a
  model that Bedrock serves everywhere it was checked.
- **`INFERENCE_PROFILE` is a code change, not a footnote.** The bare id is refused, and
  the profile prefix that works is Region-specific, so hard-coding `us.` breaks the
  first time someone runs your code in Europe.
- **"Supported" is a property of a model and an API together.** Responses support
  splits within this family, model by model; `temperature`, `top_p` and prompt caching
  each answer differently for the same model depending on the surface you send them on.
- **Read `usage`, not the status code**, for anything whose benefit is a discount.